# Ollama 泰卢固语（Tenglish）网站摘要

## 练习目标（理念）

使用**本地 Ollama** 模型，把任意网页总结成用**英文字母拼写的泰卢固语**（常称 **Tenglish**）——不调用付费云端 API。

这是对课程 **Day 1「网页摘要」** 的社区变体：抓取网页 → 组装 `messages` → 调 Chat Completions；差别在于后端是本地 Ollama，且 system prompt 要求 Tenglish 输出。

## 使用的模型

- 模型 id：`deepseek-r1:1.5b`（经 Ollama 的 OpenAI 兼容接口调用）

## 怎么跑（先决条件）

1. [安装 Ollama](https://ollama.com) 并确保本地服务在跑（默认 `http://localhost:11434`）
2. 拉取模型：`ollama pull deepseek-r1:1.5b`
3. 从上到下依次运行本笔记本单元格（Shift+Enter）
4. 在最后几格把 URL 换成你想摘要的站点


In [ ]:
# ========== 导入：网页抓取 + 本地 OpenAI 兼容客户端 ==========

# 导入标准库 os：读环境变量（本练习主要靠本地 Ollama，未必用到密钥）
import os
# 从 dotenv 导入 load_dotenv：若有 .env 可统一加载（本笔记本后续未强制调用，保留导入以兼容环境）
from dotenv import load_dotenv
# 从 bs4 导入 BeautifulSoup：解析 HTML，抽出可见文本
from bs4 import BeautifulSoup
# 导入 requests：用 HTTP GET 拉取网页
import requests
# 从 IPython.display 导入展示工具：在笔记本里渲染 Markdown 摘要
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：这里会把它的 base_url 指到本地 Ollama
from openai import OpenAI


In [ ]:
# ========== Ollama 客户端：指向本机 OpenAI 兼容端点 ==========

# base_url 指向 Ollama 的 /v1；api_key 可为任意非空字符串（本地通常不校验）
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# 模型名必须与 ollama pull / ollama list 里一致
MODEL = "deepseek-r1:1.5b"


In [ ]:
# ========== 网页爬虫：GET → 去噪 → 截断正文 ==========

# 伪装成常见浏览器 User-Agent，降低被站点简单拦截的概率
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url):
    """抓取 url 的标题+正文文本，去掉 script/style 等，并截断到约 2000 字符。"""
    # 发起 HTTP GET；带上 headers
    response = requests.get(url, headers=headers)
    # 用 html.parser 把响应字节解析成可查询的 DOM 树
    soup = BeautifulSoup(response.content, "html.parser")
    # 取 <title>；没有标题则用占位英文串（影响模型上下文，保持原样）
    title = soup.title.string if soup.title else "No title found"
    # 有 <body> 才抽正文
    if soup.body:
        # 删除无关标签：脚本、样式、图片、输入框，避免噪声进 prompt
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 抽出纯文本；换行分隔、去掉首尾空白
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        # 无 body：正文置空
        text = ""
    # 标题与正文拼接后截断，控制 prompt 长度（本地小模型上下文有限）
    return (title + "\n\n" + text)[:2_000]


In [ ]:
# ========== 提示词：要求用英文字母写泰卢固语（Tenglish） ==========

# system prompt 保留英文：规定输出语言/格式；改译会改变模型行为
system_prompt = """
You are a helpful assistant that summarizes websites in Telugu language using English letters (Tenglish).
Do NOT use Telugu script. Write Telugu words phonetically in English letters only.
Respond in markdown format.
"""

def messages_for(website):
    """把抓到的网页文本包装成 system + user 的 messages 列表。"""
    return [
        {"role": "system", "content": system_prompt},
        # user：摘要指令 + 网页正文；发给模型的英文指令保持原样
        {"role": "user",   "content": f"Summarize this website. If it has news or announcements, include those too.\n\n{website}"}
    ]


In [ ]:
# ========== 摘要函数：抓取 → 调本地模型 → Markdown 展示 ==========

def summarize(url):
    """对给定 URL 做本地模型摘要，并在笔记本中 display 为 Markdown。"""
    # 先抓网页文本
    website = fetch_website_contents(url)
    # 经 OpenAI 兼容接口调用 Ollama 上的 MODEL
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(website)
    )
    # 取出助手回复内容，用 Markdown 漂亮显示
    display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 试跑：把任意 URL 总结成 Tenglish ==========

# 调用摘要；URL 保持原样（可换成你感兴趣的站点）
summarize("https://jcpenney.com")


In [ ]:
# ========== 再试一个新闻站点 ==========

# 再跑一次不同域名，对比摘要风格与稳定性
summarize("https://bbc.com")
